# Notebook 31: Standard Model Gauge Group from the Polygon Hierarchy

**Paper IV, Sections 6--8.** Verifies the emergence of SU(3) x SU(2) x U(1) from
the Havelock polygon hierarchy at N=7. Six computations:

1. Frobenius orbits at N=7 (quadratic residues/non-residues mod 7)
2. McKay correspondence: Z/3 -> SU(2) -> A_2 = SU(3)
3. SU(3) from the Klein quartic: 3 x 3-bar = 1 + 8 (Eightfold Way)
4. Three generations from (N-1)/2 = 3 palindromic pairs
5. Confinement: 1 + omega + omega^2 = 0 (Z_3 character sum)
6. String tension: sigma/Lambda^2 = 5.97 vs lattice 6.25 +/- 0.5

In [ ]:
import sys
sys.path.insert(0, '../src')

import math
import cmath
from math import pi, sqrt, sin, cos, log, exp, gcd

from planetary_polygons.extensions.standard_model_gauge import (
    frobenius_orbits, mckay_embedding, su3_from_mckay,
    weinberg_angle_cs_threshold, casimir, spin_j,
    b_exact, central_charge, standard_model_table
)

assertion_count = 0
def check(condition, msg):
    global assertion_count
    assert condition, f'FAILED: {msg}'
    assertion_count += 1
    print(f'  [ok] {msg}')

## 1. Frobenius Orbits at N=7

The Frobenius automorphism sigma: m -> 2m mod 7 generates Z/3Z in Aut(Z/7Z)*.
Since 2^3 = 8 = 1 mod 7, sigma has order 3.

The two orbits are the quadratic residues and non-residues mod 7.

In [ ]:
N = 7
orbits = frobenius_orbits(N)
print(f'Frobenius orbits at N={N} (sigma: m -> 2m mod {N}):')
for i, orb in enumerate(orbits):
    casimirs = [casimir(m, N) for m in orb]
    print(f'  O_{i+1} = {orb}  Casimirs: {casimirs}')

# Identify which orbit contains 1
orbit_plus = [orb for orb in orbits if 1 in orb][0]
orbit_minus = [orb for orb in orbits if 1 not in orb][0]

print(f'\nO+ (containing 1) = {orbit_plus}')
print(f'O- (complement)   = {orbit_minus}')

# Verify these are QR / QNR mod 7
qr_7 = sorted({(m*m) % 7 for m in range(1, 7)})
qnr_7 = sorted(set(range(1, 7)) - set(qr_7))
print(f'\nQuadratic residues mod 7:     {qr_7}')
print(f'Quadratic non-residues mod 7: {qnr_7}')

check(sorted(orbit_plus) == qr_7, 'O+ = quadratic residues mod 7')
check(sorted(orbit_minus) == qnr_7, 'O- = quadratic non-residues mod 7')
check(len(orbits) == 2, 'Exactly 2 orbits')
check(len(orbit_plus) == 3, 'O+ has 3 elements (Z/3Z action)')
check(len(orbit_minus) == 3, 'O- has 3 elements (Z/3Z action)')

# Verify identical Casimir multisets
cas_plus = sorted([casimir(m, N) for m in orbit_plus])
cas_minus = sorted([casimir(m, N) for m in orbit_minus])
print(f'\nCasimir multiset O+: {cas_plus}')
print(f'Casimir multiset O-: {cas_minus}')
check(cas_plus == cas_minus, 'Both orbits have identical Casimir multisets')

## 2. McKay Correspondence: Z/3 -> SU(2) -> A_2 = SU(3)

The Frobenius sigma acts on O+ = {1,2,4} by cyclic permutation.
Diagonalising: the charged modes carry diag(omega, omega^{-1}) in SU(2).
By the McKay correspondence (McKay 1980): Z/3Z subset SU(2) -> A_2 Dynkin diagram -> SU(3).

In [ ]:
mckay = mckay_embedding(N)

print(f'McKay embedding at N={N}:')
print(f'  Frobenius generator: sigma: m -> {mckay["frobenius_generator"]}m mod {N}')
print(f'  Order of sigma: {mckay["frobenius_order"]}')
print(f'  Orbit O+: {mckay["orbit_plus"]}')
print(f'  Orbit O-: {mckay["orbit_minus"]}')
print(f'  omega = e^(2pi i/3) = {mckay["omega"]:.6f}')
print(f'  McKay embedding: {mckay["mckay_embedding"]}')
print(f'  det check (det(diag(omega, omega^-1)) = 1): {mckay["det_check"]}')
print(f'  Dynkin diagram: {mckay["dynkin_diagram"]}')
print(f'  Gauge group: {mckay["gauge_group"]}')
print(f'  Level: {mckay["mckay_level"]}')
print(f'  Current algebra: {mckay["current_algebra"]}')
print(f'  c(SU(3)_1) = {mckay["central_charge_su3"]}')

omega = cmath.exp(2j * cmath.pi / 3)
det = omega * omega.conjugate()  # omega * omega^{-1} = 1
check(abs(det - 1.0) < 1e-10, 'diag(omega, omega^-1) has det = 1 (SU(2))')
check(mckay['frobenius_order'] == 3, 'sigma has order 3 (2^3 = 1 mod 7)')
check(mckay['dynkin_diagram'] == 'A_2' or mckay['dynkin_diagram'] == 'A\u2082',
      'McKay maps Z/3Z to A_2 Dynkin diagram')
check(mckay['gauge_group'] == 'SU(3)', 'A_2 = SU(3)')

# Verify c(SU(3)_1) = dim(SU(3)) * k / (k + h_dual) = 8 * 1 / (1 + 3) = 2
c_su3_1 = 8 * 1 / (1 + 3)
print(f'\nc(SU(3)_1) = dim * k / (k + h_dual) = 8 * 1 / (1 + 3) = {c_su3_1}')
check(abs(c_su3_1 - 2.0) < 1e-10, 'c(SU(3)_1) = 2')

## 3. SU(3) from the Klein Quartic: 3 x 3-bar = 1 + 8

PSL(2, F_7) embeds faithfully in SU(3) via the holomorphic differentials
of the Klein quartic X(7). The tensor product of the fundamental and
antifundamental representations decomposes as 3 x 3-bar = 1 + 8
(the Eightfold Way).

In [ ]:
su3 = su3_from_mckay(N)

print('SU(3) from McKay correspondence at N=7:')
print(f'  Method: {su3["method"]}')
print(f'  Gauge group: {su3["gauge_group"]}')
print(f'  Level: {su3["level"]}')
print()
print('Proof chain:')
for step in su3['proof_chain']:
    print(f'  {step}')

# The Eightfold Way: 3 x 3-bar = 1 + 8
print('\nRepresentation decomposition (SU(3)):')
print('  3 x 3-bar = 1 + 8  (adjoint = Eightfold Way)')
dim_fund = 3
dim_adjoint = 8
check(dim_fund * dim_fund == 1 + dim_adjoint, '3 x 3-bar = 1 + 8')
check(su3['gauge_group'] == 'SU(3)', 'Gauge group is SU(3)')
check(su3['level'] == 1, 'CS level is k=1')

## 4. Three Generations: (N-1)/2 = 3 Palindromic Pairs

For N=7, the modes m=1,...,6 pair into 3 palindromic pairs (m, 7-m).
Each pair has the same Casimir: f(m,7) = f(7-m,7).

In [ ]:
n_gen = (N - 1) // 2
print(f'N = {N}: number of generations = (N-1)/2 = {n_gen}')

pairs = [(m, N - m) for m in range(1, n_gen + 1)]
print(f'\nPalindromic pairs:')
print(f'{"Gen":>5} {"Pair":>10} {"f(m,N)":>10} {"f(N-m,N)":>12} {"Equal?":>8}')
print('-' * 50)
for gen, (m1, m2) in enumerate(pairs, 1):
    f1 = casimir(m1, N)
    f2 = casimir(m2, N)
    eq = abs(f1 - f2) < 1e-10
    print(f'{gen:5d} ({m1},{m2}){"":>5} {f1:10.1f} {f2:12.1f} {"yes" if eq else "NO":>8}')

check(n_gen == 3, '(N-1)/2 = 3 generations')
check(pairs == [(1, 6), (2, 5), (3, 4)], 'Pairs are (1,6), (2,5), (3,4)')
for m1, m2 in pairs:
    check(abs(casimir(m1, N) - casimir(m2, N)) < 1e-10,
          f'f({m1},{N}) = f({m2},{N}) = {casimir(m1,N)}')

## 5. Confinement: Z_3 Character Sum and Center Symmetry

The Wilson loop in the fundamental representation of SU(3)_1 vanishes
on any genus-g surface, by the Verlinde formula:

W_fund proportional to 1 + omega + omega^2 = 0

where omega = e^(2pi i/3) is a primitive cube root of unity.
This is exact topological confinement.

The center symmetry argument: the Polyakov loop transforms as
P -> omega * P under Z_3. Since gcd(7,3) = 1, the Z_3 center
is unbroken in the polygon phase, forcing <P> = 0.

In [ ]:
print('=== Z_3 character sum (confinement) ===')
omega = cmath.exp(2j * cmath.pi / 3)
print(f'omega = e^(2pi i/3) = {omega}')
print(f'omega^2 = {omega**2}')

char_sum = 1 + omega + omega**2
print(f'\n1 + omega + omega^2 = {char_sum}')
print(f'|1 + omega + omega^2| = {abs(char_sum):.2e}')

check(abs(char_sum) < 1e-10, '1 + omega + omega^2 = 0 (Z_3 character sum)')

print(f'\n=== Center symmetry ===')
print(f'gcd(N, 3) = gcd({N}, 3) = {gcd(N, 3)}')
check(gcd(N, 3) == 1, 'gcd(7, 3) = 1 (Z_3 center unbroken)')

print(f'\nSince gcd({N}, 3) = 1:')
print(f'  The Z_{N} polygon symmetry is coprime to the Z_3 center.')
print(f'  The center symmetry is unbroken -> <P> = 0 -> confinement.')

# Verify for which N the center is unbroken
print(f'\nCenter symmetry check for various N:')
for n in range(3, 16):
    g = gcd(n, 3)
    status = 'UNBROKEN (confined)' if g == 1 else 'broken (deconfined)'
    print(f'  N={n:2d}: gcd({n},3) = {g}  -> center {status}')

## 6. String Tension: sigma/Lambda^2 = 5.97

The physical string tension is:

sigma/Lambda^2 = sigma_YM x dim(H) / Z(S^3)^2

where:
- dim H(Sigma_2, SU(3)_1) = 9 (Verlinde formula)
- Z(S^3)^2 = 2 (Witten formula for SU(3) at k=1)
- sigma_YM = 1.326 (bare YM string tension on H^2)

In [ ]:
print('=== Verlinde dimension: dim H(Sigma_2, SU(3)_1) ===')
print()
# SU(3)_1 has 3 integrable representations (trivial, fund, antifund)
# S-matrix: S_{0,lambda} = 1/sqrt(3) for all lambda
n_reps = 3  # number of integrable representations of SU(3)_1
S_0_lambda = 1.0 / sqrt(3)
g = 2  # genus of the Bolza surface

# Verlinde formula: dim H = sum_lambda (S_{0,lambda})^{2-2g}
dim_H = n_reps * S_0_lambda**(2 - 2*g)
print(f'  SU(3)_1: {n_reps} integrable representations')
print(f'  S_{{0,lambda}} = 1/sqrt(3) = {S_0_lambda:.6f}')
print(f'  Genus g = {g}')
print(f'  dim H = {n_reps} x (1/sqrt(3))^{{2-2*{g}}} = {n_reps} x (1/sqrt(3))^{{{2-2*g}}}')
print(f'        = {n_reps} x (sqrt(3))^{2*g-2} = {n_reps} x 3^{g-1} = {n_reps} x {3**(g-1)} = {dim_H:.0f}')

check(abs(dim_H - 9) < 1e-10, 'dim H(Sigma_2, SU(3)_1) = 9')

In [ ]:
print('=== Witten formula: Z(S^3, SU(3), k=1) = sqrt(2) ===')
print()
# Witten's formula for SU(N) at level k on S^3:
# Z(S^3) = sqrt(2/(k+N))^N * prod_{j=1}^{N-1} (2 sin(pi j/(k+N)))^{N-j}
#
# For SU(3), k=1: k+N = 4
k_cs = 1
N_gauge = 3  # SU(3)
kpN = k_cs + N_gauge  # = 4

print(f'SU({N_gauge}) at level k={k_cs}: k+N = {kpN}')
print()

# Prefactor: sqrt(2/(k+N))^N = sqrt(2/4)^3 = (1/sqrt(2))^3 = 1/(2 sqrt(2))
prefactor = (sqrt(2.0 / kpN)) ** N_gauge
print(f'Prefactor: sqrt(2/{kpN})^{N_gauge} = sqrt({2.0/kpN:.4f})^{N_gauge} = {prefactor:.6f}')
print(f'         = 1/(2 sqrt(2)) = {1/(2*sqrt(2)):.6f}')

check(abs(prefactor - 1/(2*sqrt(2))) < 1e-10, 'Prefactor = 1/(2 sqrt(2))')

# Product: j=1 term: (2 sin(pi/4))^{3-1} = (2 * sqrt(2)/2)^2 = (sqrt(2))^2 = 2
j1_term = (2 * sin(pi * 1 / kpN)) ** (N_gauge - 1)
print(f'\nj=1: (2 sin(pi*1/{kpN}))^{{{N_gauge}-1}} = (2 sin(pi/4))^2 = (2 * {sin(pi/4):.4f})^2 = {j1_term:.6f}')
check(abs(j1_term - 2.0) < 1e-10, 'j=1 term = (2 sin(pi/4))^2 = 2')

# Product: j=2 term: (2 sin(pi*2/4))^{3-2} = (2 sin(pi/2))^1 = 2^1 = 2
j2_term = (2 * sin(pi * 2 / kpN)) ** (N_gauge - 2)
print(f'j=2: (2 sin(pi*2/{kpN}))^{{{N_gauge}-2}} = (2 sin(pi/2))^1 = (2 * {sin(pi/2):.4f})^1 = {j2_term:.6f}')
check(abs(j2_term - 2.0) < 1e-10, 'j=2 term = (2 sin(pi/2))^1 = 2')

# Full partition function
Z_S3 = prefactor * j1_term * j2_term
print(f'\nZ(S^3) = {prefactor:.6f} x {j1_term:.6f} x {j2_term:.6f} = {Z_S3:.6f}')
print(f'       = 1/(2 sqrt(2)) x 2 x 2 = {1/(2*sqrt(2)) * 4:.6f}')
print(f'       = sqrt(2) = {sqrt(2):.6f}')

check(abs(Z_S3 - sqrt(2)) < 1e-10, 'Z(S^3, SU(3), k=1) = sqrt(2)')
check(abs(Z_S3**2 - 2.0) < 1e-10, 'Z(S^3)^2 = 2')

In [ ]:
print('=== String tension: sigma/Lambda^2 = 5.97 ===')
print()

# Bare YM string tension (from paper)
sigma_polygon = 0.1015   # geometric (Havelock eigenvalue at WDW ground state)
sigma_polyakov = 0.612   # monopole (Polyakov mechanism)
n_monopole = 2           # rank(SU(3)) = 2 independent monopole species

sigma_YM = sigma_polygon + n_monopole * sigma_polyakov
print(f'sigma_polygon  = {sigma_polygon}')
print(f'sigma_Polyakov = {sigma_polyakov} (per monopole species)')
print(f'rank(SU(3))    = {n_monopole} (independent monopole species)')
print(f'sigma_YM       = {sigma_polygon} + {n_monopole} x {sigma_polyakov} = {sigma_YM}')

# Non-perturbative CS enhancement
ratio = dim_H / Z_S3**2
print(f'\ndim H / Z(S^3)^2 = {dim_H:.0f} / {Z_S3**2:.0f} = {ratio}')

# Final result
sigma_total = sigma_YM * ratio
print(f'\nsigma/Lambda^2 = sigma_YM x dim(H)/Z(S^3)^2')
print(f'               = {sigma_YM} x {ratio}')
print(f'               = {sigma_total}')
print(f'\nLattice value: 6.25 +/- 0.5')
print(f'Match: {abs(sigma_total - 6.25)/6.25 * 100:.1f}% off (within 1 sigma)')

check(abs(sigma_total - 5.97) < 0.01, 'sigma/Lambda^2 = 5.97')
check(abs(sigma_total - 6.25) < 0.5, 'sigma/Lambda^2 within lattice error bars (6.25 +/- 0.5)')

## Summary

In [ ]:
print(f'All {assertion_count} assertions passed.')